# 03 — localized pages (fixture site build)

Builds the fixture site (netsnek.com on its jaen feature branch) and
inspects the generated pages: one variant per locale, correct
`<html lang>`, canonical links, unlocalized system routes.

The build is expensive; set `JAEN_SKIP_SITE_BUILD=1` to reuse an
existing `public/`.


In [ ]:
import jaen_testkit as k
k.start_run('03-pages-i18n')
print(k.CONFIG['repo_root'])

In [ ]:
import os

SITE = k.CONFIG['site_dir']
PUBLIC = k.site_path('public')

with k.section('site build'):
    with k.check('fixture site builds') as c:
        if not os.path.isdir(SITE):
            c.skip('no fixture site checkout')
        if os.environ.get('JAEN_SKIP_SITE_BUILD') == '1':
            if os.path.isdir(PUBLIC):
                c.skip('JAEN_SKIP_SITE_BUILD=1 — reusing existing public/')
            c.skip('JAEN_SKIP_SITE_BUILD=1 but no public/ present')
        r = c.require(k.sh('yarn build', cwd=SITE,
                           timeout=k.CONFIG['build_timeout'],
                           label='gatsby build (site)'))
        c.ok('built in %.0fs' % r.duration_s)


In [ ]:
locales = k.CONFIG['site_locales']
default = k.CONFIG['site_default_locale']

with k.section('localized variants'):
    with k.check('index page exists per locale') as c:
        if not os.path.isdir(PUBLIC):
            c.skip('no public/ — build did not run')
        for locale in locales:
            path = (os.path.join(PUBLIC, 'index.html') if locale == default
                    else os.path.join(PUBLIC, locale, 'index.html'))
            c.expect_true(os.path.isfile(path), '%s -> %s' % (locale, os.path.relpath(path, PUBLIC)))

    with k.check('html lang matches the locale') as c:
        if not os.path.isdir(PUBLIC):
            c.skip('no public/')
        import re
        for locale in locales:
            path = (os.path.join(PUBLIC, 'index.html') if locale == default
                    else os.path.join(PUBLIC, locale, 'index.html'))
            html = k.read_text(path, '')
            m = re.search(r'<html[^>]*\blang=\"([^\"]+)\"', html)
            got = m.group(1) if m else None
            c.expect_true(bool(got) and got.lower().startswith(locale.split('-')[0]),
                          '%s: lang=%s' % (locale, got))


In [ ]:
with k.section('canonical + system routes'):
    with k.check('canonical link is absolute and normalized') as c:
        if not os.path.isdir(PUBLIC):
            c.skip('no public/')
        import re
        html = k.read_text(os.path.join(PUBLIC, 'index.html'), '')
        m = re.search(r'<link[^>]*rel=\"canonical\"[^>]*href=\"([^\"]+)\"', html)
        if not m:
            c.fail('no canonical link on the index page', abort=True)
        href = m.group(1)
        c.expect_true(href.startswith('https://'), href)
        c.expect_true('//' not in href.split('://', 1)[1], 'no double slashes: %s' % href)

    with k.check('system routes are not localized') as c:
        if not os.path.isdir(PUBLIC):
            c.skip('no public/')
        offenders = []
        for locale in locales:
            if locale == default:
                continue
            for system in ('cms', 'login', 'logout', 'settings', 'signup'):
                candidate = os.path.join(PUBLIC, locale, system)
                if os.path.isdir(candidate):
                    offenders.append('%s/%s' % (locale, system))
        c.expect_equal(offenders, [], 'no /<locale>/<system> directories')


In [ ]:
k.summary()
k.save_results('results-03-pages-i18n.json')
rc = k.verdict()
assert rc == 0, 'run has FAILures — see the summary above'